# Pretrained

In [1]:
!pip install -q sentence-transformers lightgbm wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.5 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [2]:
import os, glob, warnings
import numpy as np
import pandas as pd
import wandb
import torch
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import SentenceTransformer
warnings.filterwarnings('ignore')

OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ── Load data ──────────────────────────────────────────────────────────────
train_path = 'train.csv'
test_path  = 'test.csv'

if not os.path.exists(train_path):
    found = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if found:
        train_path = found[0]
        test_path  = glob.glob('/kaggle/input/**/test.csv', recursive=True)[0]

df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)
print(f'Train: {df_train.shape}  |  Test: {df_test.shape}')

Device: cuda
Train: (2000, 8)  |  Test: (500, 7)


In [3]:


wandb.login()

run = wandb.init(
    project='smart-mcq-solver',
    name='semantic-ensemble-v1',
    config={
        'model'                   : 'all-mpnet-base-v2 + LightGBM + HistGBM + ExtraTrees',
        'sentence_transformer'    : 'all-mpnet-base-v2',
        'n_folds'                 : 5,
        'lgb_num_leaves'          : 63,
        'lgb_learning_rate'       : 0.05,
        'lgb_num_boost_round'     : 500,
        'extra_trees_n_estimators': 200,
        'tfidf_word_ngram'        : '(1,2)',
        'tfidf_char_ngram'        : '(3,5)',
        'train_size'              : len(df_train),
        'test_size'               : len(df_test)
    }
)
print(f'WandB run initialized: {run.name}  |  URL: {run.url}')

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


WandB run initialized: semantic-ensemble-v1  |  URL: https://wandb.ai/vinodparvathy-indian-institute-of-technology-madras/smart-mcq-solver/runs/r9nirowy


In [4]:
# ── Load Sentence Transformer ──────────────────────────────────────────────
print('Loading sentence transformer model...')
st_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device=DEVICE)
print('Model loaded!')

Loading sentence transformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded!


In [5]:
# ── Encode prompts and options for semantic similarity ─────────────────────
def get_semantic_sims(df):
    prompts = df['prompt'].fillna('').astype(str).tolist()
    print(f'  Encoding {len(prompts)} prompts...')
    prompt_embs = st_model.encode(prompts, batch_size=64, show_progress_bar=True,
                                  convert_to_numpy=True, normalize_embeddings=True)
    sims = np.zeros((len(df), 5), dtype=np.float32)
    for i, col in enumerate(OPTION_COLS):
        opts = df[col].fillna('').astype(str).tolist()
        print(f'  Encoding option {col}...')
        opt_embs = st_model.encode(opts, batch_size=64, show_progress_bar=False,
                                   convert_to_numpy=True, normalize_embeddings=True)
        sims[:, i] = (prompt_embs * opt_embs).sum(axis=1)
    return sims

print('Computing semantic similarities for train...')
train_sims = get_semantic_sims(df_train)
print('Computing semantic similarities for test...')
test_sims  = get_semantic_sims(df_test)
print(f'train_sims: {train_sims.shape}  |  test_sims: {test_sims.shape}')

Computing semantic similarities for train...
  Encoding 2000 prompts...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

  Encoding option A...
  Encoding option B...
  Encoding option C...
  Encoding option D...
  Encoding option E...
Computing semantic similarities for test...
  Encoding 500 prompts...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

  Encoding option A...
  Encoding option B...
  Encoding option C...
  Encoding option D...
  Encoding option E...
train_sims: (2000, 5)  |  test_sims: (500, 5)


In [6]:
# ── Build TF-IDF features (train+test corpus for transductive trick) ───────
def build_corpus(df):
    texts = df['prompt'].astype(str).tolist()
    for c in OPTION_COLS:
        texts += df[c].fillna('').astype(str).tolist()
    return texts

corpus   = build_corpus(df_train) + build_corpus(df_test)
word_vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), sublinear_tf=True, min_df=2)
char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True, min_df=3)
word_vec.fit(corpus)
char_vec.fit(corpus)

def extract_features(df, sem_sims):
    rows = []
    for row_i, (_, row) in enumerate(df.iterrows()):
        prompt       = str(row['prompt'])
        prompt_words = set(prompt.lower().split())
        pw_vec       = word_vec.transform([prompt])
        pc_vec       = char_vec.transform([prompt])
        opts         = [str(row[c]) for c in OPTION_COLS]
        ow_vecs      = word_vec.transform(opts)
        oc_vecs      = char_vec.transform(opts)
        word_sims    = cosine_similarity(pw_vec, ow_vecs)[0]
        char_sims    = cosine_similarity(pc_vec, oc_vecs)[0]
        lengths      = np.array([len(o) for o in opts], dtype=float)
        avg_len      = lengths.mean()
        max_len      = lengths.max()
        for i, opt in enumerate(opts):
            opt_words = set(opt.lower().split())
            inter     = prompt_words & opt_words
            union     = prompt_words | opt_words
            rows.append([
                sem_sims[row_i, i],
                word_sims[i],
                char_sims[i],
                lengths[i],
                lengths[i] / (avg_len + 1e-9),
                float(lengths[i] == max_len),
                len(inter),
                len(inter) / (len(opt_words) + 1e-9),
                len(inter) / (len(union) + 1e-9),
            ])
    return np.array(rows, dtype=np.float32)

print('Extracting combined features...')
X_all      = extract_features(df_train, train_sims)
y_all      = np.array([1.0 if str(row['answer']) == c else 0.0
                       for _, row in df_train.iterrows() for c in OPTION_COLS])
X_test_all = extract_features(df_test, test_sims)
print(f'Train features: {X_all.shape}  |  Test features: {X_test_all.shape}')

Extracting combined features...
Train features: (10000, 9)  |  Test features: (2500, 9)


In [7]:
# ── Helper functions 
def get_top3(probs, n_questions):
    preds = []
    for i in range(n_questions):
        g = probs[i*5:(i+1)*5]
        ranked = np.argsort(g)[::-1]
        preds.append([OPTION_COLS[r] for r in ranked[:3]])
    return preds

def map_at_3(preds, targets):
    score = 0.0
    for pred, target in zip(preds, targets):
        for i, p in enumerate(pred):
            if p == target:
                score += 1.0 / (i + 1)
                break
    return score / len(targets)

In [8]:
# ── 5-Fold CV with LightGBM + HistGBM + ExtraTrees Ensemble ───────────────
skf              = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores      = []
fold_accs        = []
fold_f1s         = []
test_preds_folds = []

print('\n--- 5-Fold Cross-Validation ---')
for fold, (tr_q, va_q) in enumerate(skf.split(df_train, df_train['answer'])):
    tr_idx = np.concatenate([np.arange(q*5, q*5+5) for q in tr_q])
    va_idx = np.concatenate([np.arange(q*5, q*5+5) for q in va_q])

    X_tr, y_tr = X_all[tr_idx], y_all[tr_idx]
    X_va       = X_all[va_idx]

    # LightGBM
    lgb_tr = lgb.Dataset(X_tr, y_tr)
    lgb_va = lgb.Dataset(X_va, y_all[va_idx], reference=lgb_tr)
    params = dict(objective='binary', metric='binary_logloss',
                  num_leaves=63, learning_rate=0.05, n_jobs=-1, verbose=-1, seed=42)
    m_lgb  = lgb.train(params, lgb_tr, num_boost_round=500,
                       valid_sets=[lgb_va],
                       callbacks=[lgb.early_stopping(40, verbose=False),
                                  lgb.log_evaluation(period=-1)])
    p_lgb  = m_lgb.predict(X_va)

    # HistGBM
    m_hgb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05,
                                           max_leaf_nodes=63, random_state=42)
    m_hgb.fit(X_tr, y_tr)
    p_hgb = m_hgb.predict_proba(X_va)[:, 1]

    # ExtraTrees
    m_et = ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    m_et.fit(X_tr, y_tr)
    p_et  = m_et.predict_proba(X_va)[:, 1]

    p_ens = (p_lgb + p_hgb + p_et) / 3.0

    val_targets = [df_train.iloc[q]['answer'] for q in va_q]
    top3_preds  = get_top3(p_ens, len(va_q))
    top1_preds  = [p[0] for p in top3_preds]

    score     = map_at_3(top3_preds, val_targets)
    fold_acc  = accuracy_score(val_targets, top1_preds)
    fold_f1   = f1_score(val_targets, top1_preds, average='macro', zero_division=0)

    fold_scores.append(score)
    fold_accs.append(fold_acc)
    fold_f1s.append(fold_f1)

    print(f'  Fold {fold+1}  MAP@3: {score:.5f}  |  Accuracy: {fold_acc:.5f}  |  F1: {fold_f1:.5f}')

    # ── Log per-fold metrics to WandB ──────────────────────────────────────
    wandb.log({
        'fold'          : fold + 1,
        'fold_map@3'    : score,
        'fold_accuracy' : fold_acc,
        'fold_f1_macro' : fold_f1,
    })

    p_test = (m_lgb.predict(X_test_all) +
              m_hgb.predict_proba(X_test_all)[:, 1] +
              m_et.predict_proba(X_test_all)[:, 1]) / 3.0
    test_preds_folds.append(p_test)

mean_map3 = np.mean(fold_scores)
mean_acc  = np.mean(fold_accs)
mean_f1   = np.mean(fold_f1s)
print(f'\nMean CV  MAP@3: {mean_map3:.5f}  |  Accuracy: {mean_acc:.5f}  |  F1: {mean_f1:.5f}')

# ── Log final summary to WandB ─────────────────────────────────────────────
wandb.summary['mean_map@3']      = mean_map3
wandb.summary['mean_accuracy']   = mean_acc
wandb.summary['mean_f1_macro']   = mean_f1
wandb.summary['best_fold_map@3'] = max(fold_scores)


--- 5-Fold Cross-Validation ---
  Fold 1  MAP@3: 0.96000  |  Accuracy: 0.92500  |  F1: 0.92531
  Fold 2  MAP@3: 0.97167  |  Accuracy: 0.94500  |  F1: 0.94484
  Fold 3  MAP@3: 0.95625  |  Accuracy: 0.92250  |  F1: 0.92299
  Fold 4  MAP@3: 0.95583  |  Accuracy: 0.91750  |  F1: 0.91729
  Fold 5  MAP@3: 0.95542  |  Accuracy: 0.92250  |  F1: 0.92375

Mean CV  MAP@3: 0.95983  |  Accuracy: 0.92650  |  F1: 0.92684


In [9]:
# ── Generate Submission ────────────────────────────────────────────────────
avg_test_probs = np.mean(test_preds_folds, axis=0)
test_preds     = get_top3(avg_test_probs, len(df_test))

submission = pd.DataFrame({
    'ID'        : df_test['id'].values,
    'Prediction': [' '.join(p) for p in test_preds]
})
submission.to_csv('submission.csv', index=False)
print('submission.csv saved!')

# ── Log submission artifact to WandB and finish ────────────────────────────
artifact = wandb.Artifact('submission', type='predictions')
artifact.add_file('submission.csv')
wandb.log_artifact(artifact)
wandb.finish()
print('WandB run finished! View at https://wandb.ai')

submission.head(10)

submission.csv saved!


fold,▁▃▅▆█
fold_accuracy,▃█▂▁▂
fold_f1_macro,▃█▂▁▃
fold_map@3,▃█▁▁▁
best_fold_map@3,0.97167
fold,5
fold_accuracy,0.9225
fold_f1_macro,0.92375
fold_map@3,0.95542
mean_accuracy,0.9265
mean_f1_macro,0.92684


WandB run finished! View at https://wandb.ai


,ID,Prediction
0,1,A B D
1,2,B E D
2,3,B E C
3,4,E A C
4,5,C A D
5,6,D C B
6,7,E D B
7,8,B D C
8,9,C D B
9,10,B E C
